##12.Text Models: LSTM vs Transformer

### Architecture Justification

**LSTM — chosen for sequential pattern detection:**
- Input: padded sequences of max 100 tokens
- Embedding dim = 64 (balances vocab size 10,000 with model capacity)
- Hidden units = 32 (prevents overfitting on small dataset)
- Single layer to avoid vanishing gradient on short texts

**Transformer — chosen for global context modeling:**
- num_heads = 2 (sufficient for binary classification)
- key_dim = 32 (matches embedding size)
- GlobalAveragePooling instead of CLS token (simpler, works well on short texts)

**Why both?**
- LSTM captures local sequential patterns (word order matters)
- Transformer captures global dependencies (any word can attend to any other)
- Best model selected automatically based on validation accuracy

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import MultiHeadAttention, LayerNormalization, GlobalAveragePooling1D

# --- Model 1: LSTM ---
model_text = Sequential([
    Embedding(10000, 64),
    LSTM(32),
    Dense(1, activation='sigmoid')
])

model_text.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history_text = model_text.fit(
    X_text_train,
    y_train,
    epochs=5,
    batch_size=32,
    validation_data=(X_text_test, y_test)
)

text_pred = (model_text.predict(X_text_test) > 0.5).astype(int).ravel()
lstm_acc = accuracy_score(y_test, text_pred)
print("LSTM Accuracy:", lstm_acc)


# --- Model 2: Transformer with Positional Encoding ---
inputs = tf.keras.Input(shape=(100,))

# Token Embedding
x = Embedding(10000, 64)(inputs)

# Positional Encoding
positions = tf.range(start=0, limit=100, delta=1)
pos_embed = Embedding(100, 64)(positions)
x = x + pos_embed  # إضافة الـ positional info للـ token embeddings

# Transformer Block
x = MultiHeadAttention(num_heads=2, key_dim=32)(x, x)
x = LayerNormalization()(x)
x = GlobalAveragePooling1D()(x)
x = Dense(32, activation='relu')(x)
outputs = Dense(1, activation='sigmoid')(x)

model_text_transformer = tf.keras.Model(inputs, outputs)

model_text_transformer.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history_transformer = model_text_transformer.fit(
    X_text_train,
    y_train,
    epochs=5,
    batch_size=32,
    validation_data=(X_text_test, y_test)
)

transformer_pred = (model_text_transformer.predict(X_text_test) > 0.5).astype(int).ravel()
transformer_acc = accuracy_score(y_test, transformer_pred)
print("Transformer Accuracy:", transformer_acc)


# --- Comparison ---
print("\n=== Text Model Comparison ===")
print("LSTM Accuracy:        " + str(round(lstm_acc, 4)))
print("Transformer Accuracy: " + str(round(transformer_acc, 4)))

if transformer_acc > lstm_acc:
    print("Transformer ahsan - hanstakhdemoh fel Fusion")
    best_text_pred = transformer_pred
else:
    print("LSTM ahsan - hanstakhdemoh fel Fusion")
    best_text_pred = text_pred